# Hyperparameter Tuning for Quantum Material Band-Gap Prediction

This notebook investigates whether hyperparameter optimization can improve
the Random Forest model developed in the previous notebook.

The feature representation and target variable are kept unchanged.

Objective:
- Optimize Random Forest hyperparameters
- Evaluate performance using cross-validation
- Compare the tuned model with the baseline enhanced Random Forest
- Evaluate the final tuned model on a held-out test set

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [5]:
data_path = Path("../data/enhanced_material_descriptors.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (93902, 26)

Columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap']


In [8]:
feature_columns = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "crys",
    "spg_number",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group"
]

X = df[feature_columns]
y = df["target_bandgap"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (93902, 25)
y shape: (93902,)


In [9]:
y_class = (y > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining zero-gap:", (y_train == 0).sum())
print("Training nonzero-gap:", (y_train > 0).sum())

print("\nTest zero-gap:", (y_test == 0).sum())
print("Test nonzero-gap:", (y_test > 0).sum())

Training samples: 75121
Test samples: 18781

Training zero-gap: 53649
Training nonzero-gap: 21472

Test zero-gap: 13413
Test nonzero-gap: 5368


In [10]:
numeric_features = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group"
]

categorical_features = [
    "crys",
    "spg_number"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [11]:
baseline_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", baseline_rf)
])

print("Baseline model created.")

Baseline model created.


In [13]:
baseline_pipeline.fit(X_train, y_train)

baseline_pred = baseline_pipeline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline Enhanced Random Forest")
print("--------------------------------")
print(f"MAE  : {baseline_mae:.4f} eV")
print(f"RMSE : {baseline_rmse:.4f} eV")
print(f"R²   : {baseline_r2:.4f}")

Baseline Enhanced Random Forest
--------------------------------
MAE  : 0.2498 eV
RMSE : 0.5676 eV
R²   : 0.8118


In [14]:
param_distributions = {
    "model__n_estimators": [150, 200, 300, 400],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": [0.5, 0.7, 1.0]
}

print("Hyperparameter search space defined.")

Hyperparameter search space defined.


In [15]:
random_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ))
    ]),
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Starting hyperparameter search...")

Starting hyperparameter search...


### RandomizedSearchCV

In [16]:
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=(y_train > 0)
)

print("Sub-training samples:", len(X_subtrain))
print("Validation samples:", len(X_val))

Sub-training samples: 60096
Validation samples: 15025


In [17]:
configs = [
    {
        "name": "baseline",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "name": "more_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "name": "leaf_1",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "name": "leaf_4",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 4,
        "max_features": 1.0
    },
    {
        "name": "depth_30",
        "n_estimators": 200,
        "max_depth": 30,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "name": "depth_20",
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "name": "features_0.7",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.7
    },
    {
        "name": "regularized",
        "n_estimators": 300,
        "max_depth": 30,
        "min_samples_split": 5,
        "min_samples_leaf": 4,
        "max_features": 0.7
    }
]

print("Number of configurations:", len(configs))

Number of configurations: 8


In [18]:
screening_results = []

for config in configs:

    print("\n" + "=" * 60)
    print("Testing:", config["name"])
    print("=" * 60)

    model = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features=config["max_features"],
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_subtrain, y_subtrain)

    predictions = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    r2 = r2_score(y_val, predictions)

    screening_results.append({
        "name": config["name"],
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "min_samples_split": config["min_samples_split"],
        "min_samples_leaf": config["min_samples_leaf"],
        "max_features": config["max_features"]
    })

    print(f"MAE  : {mae:.4f} eV")
    print(f"RMSE : {rmse:.4f} eV")
    print(f"R²   : {r2:.4f}")


Testing: baseline
MAE  : 0.2610 eV
RMSE : 0.5730 eV
R²   : 0.8046

Testing: more_trees
MAE  : 0.2608 eV
RMSE : 0.5725 eV
R²   : 0.8049

Testing: leaf_1
MAE  : 0.2498 eV
RMSE : 0.5628 eV
R²   : 0.8114

Testing: leaf_4
MAE  : 0.2801 eV
RMSE : 0.5944 eV
R²   : 0.7896

Testing: depth_30
MAE  : 0.2618 eV
RMSE : 0.5737 eV
R²   : 0.8041

Testing: depth_20
MAE  : 0.2733 eV
RMSE : 0.5853 eV
R²   : 0.7960

Testing: features_0.7
MAE  : 0.2638 eV
RMSE : 0.5747 eV
R²   : 0.8034

Testing: regularized
MAE  : 0.2825 eV
RMSE : 0.5944 eV
R²   : 0.7896


## Compare configurations

In [19]:
screening_df = pd.DataFrame(screening_results)

screening_df = screening_df.sort_values(
    "MAE"
).reset_index(drop=True)

print(
    screening_df[
        [
            "name",
            "MAE",
            "RMSE",
            "R2",
            "n_estimators",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features"
        ]
    ].to_string(index=False)
)

        name      MAE     RMSE       R2  n_estimators  max_depth  min_samples_split  min_samples_leaf  max_features
      leaf_1 0.249836 0.562845 0.811388           200        NaN                  2                 1           1.0
  more_trees 0.260784 0.572471 0.804881           300        NaN                  2                 2           1.0
    baseline 0.260986 0.572954 0.804552           200        NaN                  2                 2           1.0
    depth_30 0.261813 0.573661 0.804069           200       30.0                  2                 2           1.0
features_0.7 0.263802 0.574676 0.803376           200        NaN                  2                 2           0.7
    depth_20 0.273350 0.585292 0.796043           200       20.0                  2                 2           1.0
      leaf_4 0.280065 0.594411 0.789639           200        NaN                  2                 4           1.0
 regularized 0.282513 0.594420 0.789633           300       30.0        

In [20]:
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

screening_df.to_csv(
    results_path / "hyperparameter_screening.csv",
    index=False
)

print("Saved:", results_path / "hyperparameter_screening.csv")

Saved: ..\results\hyperparameter_screening.csv


In [21]:
stage2_configs = [
    {
        "name": "baseline_leaf_2",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "name": "best_leaf_1",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    }
]

stage2_configs


[{'name': 'baseline_leaf_2',
  'n_estimators': 200,
  'max_depth': None,
  'min_samples_split': 2,
  'min_samples_leaf': 2,
  'max_features': 1.0},
 {'name': 'best_leaf_1',
  'n_estimators': 200,
  'max_depth': None,
  'min_samples_split': 2,
  'min_samples_leaf': 1,
  'max_features': 1.0}]

In [24]:
def create_enhanced_preprocessor():

    numeric_features_enhanced = [
        "num_elements",
        "total_atoms",
        "mean_atomic_number",
        "min_atomic_number",
        "max_atomic_number",
        "mean_atomic_mass",
        "min_atomic_mass",
        "max_atomic_mass",
        "mean_atomic_radius",
        "min_atomic_radius",
        "max_atomic_radius",
        "mean_electronegativity",
        "min_electronegativity",
        "max_electronegativity",
        "electronegativity_difference",
        "mean_ionization_energy",
        "mean_electron_affinity",
        "mean_s_valence",
        "mean_p_valence",
        "mean_d_valence",
        "mean_f_valence",
        "mean_period",
        "mean_group"
    ]

    categorical_features_enhanced = [
        "crys",
        "spg_number"
    ]

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features_enhanced),
        ("cat", categorical_transformer, categorical_features_enhanced)
    ])

    return preprocessor

In [25]:
from sklearn.model_selection import KFold

In [26]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stage2_results = []

for config in stage2_configs:

    print(f"\nRunning: {config['name']}")

    mae_scores = []
    rmse_scores = []
    r2_scores = []

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(X_train), start=1
    ):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = RandomForestRegressor(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            min_samples_split=config["min_samples_split"],
            min_samples_leaf=config["min_samples_leaf"],
            max_features=config["max_features"],
            random_state=42,
            n_jobs=-1
        )

        pipeline = Pipeline([
            ("preprocessor", create_enhanced_preprocessor()),
            ("model", model)
        ])

        pipeline.fit(X_fold_train, y_fold_train)

        predictions = pipeline.predict(X_fold_val)

        mae = mean_absolute_error(y_fold_val, predictions)
        rmse = np.sqrt(mean_squared_error(y_fold_val, predictions))
        r2 = r2_score(y_fold_val, predictions)

        mae_scores.append(mae)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

        print(
            f"Fold {fold}: "
            f"MAE={mae:.4f}, "
            f"RMSE={rmse:.4f}, "
            f"R²={r2:.4f}"
        )

    stage2_results.append({
        "name": config["name"],
        "MAE_mean": np.mean(mae_scores),
        "MAE_std": np.std(mae_scores),
        "RMSE_mean": np.mean(rmse_scores),
        "RMSE_std": np.std(rmse_scores),
        "R2_mean": np.mean(r2_scores),
        "R2_std": np.std(r2_scores)
    })


Running: baseline_leaf_2
Fold 1: MAE=0.2608, RMSE=0.5702, R²=0.7990
Fold 2: MAE=0.2667, RMSE=0.5888, R²=0.8019
Fold 3: MAE=0.2621, RMSE=0.5822, R²=0.8049
Fold 4: MAE=0.2632, RMSE=0.5871, R²=0.8045
Fold 5: MAE=0.2620, RMSE=0.5741, R²=0.8092

Running: best_leaf_1
Fold 1: MAE=0.2490, RMSE=0.5585, R²=0.8072
Fold 2: MAE=0.2533, RMSE=0.5745, R²=0.8114
Fold 3: MAE=0.2505, RMSE=0.5733, R²=0.8109
Fold 4: MAE=0.2526, RMSE=0.5806, R²=0.8088
Fold 5: MAE=0.2501, RMSE=0.5631, R²=0.8164


In [27]:
stage2_df = pd.DataFrame(stage2_results)

stage2_df = stage2_df.sort_values(
    "MAE_mean",
    ascending=True
)

stage2_df

,name,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
1,best_leaf_1,0.25111,0.001603,0.570002,0.008066,0.810909,0.003129
0,baseline_leaf_2,0.26295,0.002011,0.580476,0.007250,0.803885,0.003395


In [28]:
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline([
    ("preprocessor", create_enhanced_preprocessor()),
    ("model", final_model)
])

print("Training final tuned model...")

final_pipeline.fit(X_train, y_train)

print("Training complete.")

Training final tuned model...
Training complete.


In [29]:
test_predictions = final_pipeline.predict(X_test)

final_mae = mean_absolute_error(y_test, test_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
final_r2 = r2_score(y_test, test_predictions)

print("FINAL TEST PERFORMANCE")
print("-" * 40)
print(f"MAE  : {final_mae:.4f} eV")
print(f"RMSE : {final_rmse:.4f} eV")
print(f"R²   : {final_r2:.4f}")

FINAL TEST PERFORMANCE
----------------------------------------
MAE  : 0.2381 eV
RMSE : 0.5559 eV
R²   : 0.8195


In [30]:
from pathlib import Path
import json

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

final_metrics = {
    "model": "RandomForestRegressor",
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "test_MAE_eV": final_mae,
    "test_RMSE_eV": final_rmse,
    "test_R2": final_r2
}

with open(results_dir / "final_model_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=4)

print("Final model metrics saved.")

Final model metrics saved.


In [31]:
test_results = X_test.copy()

test_results["actual_bandgap"] = y_test.values
test_results["predicted_bandgap"] = test_predictions
test_results["absolute_error"] = np.abs(
    test_results["actual_bandgap"] -
    test_results["predicted_bandgap"]
)

test_results.to_csv(
    results_dir / "final_test_predictions.csv",
    index=False
)

print("Test predictions saved.")

Test predictions saved.
